# Energy Recovery - ML Model Training

This notebook trains predictive models for the Energy Recovery Intelligence Agent.

## Models
1. **PX Failure Prediction** - Predict device failures within 30 days
2. **Energy Efficiency Scoring** - Score device performance vs. design specs
3. **Demand Forecasting** - Forecast product demand by region/quarter
4. **Equipment Health Scoring** - Composite health score from multiple factors

## Note
The SQL UDFs in `09_ml_model_functions.sql` use rule-based scoring that works
without trained models. This notebook is provided for future enhancement with
Snowflake ML (Cortex ML functions or Snowpark ML).

In [ ]:
# Connect to Snowflake
from snowflake.snowpark import Session
import snowflake.snowpark.functions as F

session = Session.builder.configs({
    "account": "<your_account>",
    "user": "<your_user>",
    "password": "<your_password>",
    "database": "ENERGY_RECOVERY_DB",
    "schema": "SCADA_IOT",
    "warehouse": "ENERGY_RECOVERY_WH"
}).create()

print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

In [ ]:
# Load device telemetry features for model training
telemetry_features = session.sql("""
    SELECT
        dr.DEVICE_ID,
        dr.DEVICE_MODEL,
        dr.OPERATING_HOURS,
        DATEDIFF('day', dr.LAST_SERVICE_DATE, CURRENT_DATE()) AS DAYS_SINCE_SERVICE,
        AVG(dt.VIBRATION_MM_S) AS AVG_VIBRATION,
        AVG(dt.ENERGY_RECOVERY_PCT) AS AVG_EFFICIENCY,
        AVG(dt.TEMPERATURE_C) AS AVG_TEMPERATURE,
        AVG(dt.PRESSURE_DIFFERENTIAL) AS AVG_PRESSURE_DIFF,
        COUNT(a.ALARM_ID) AS ALARM_COUNT,
        SUM(CASE WHEN a.ALARM_SEVERITY = 'Critical' THEN 1 ELSE 0 END) AS CRITICAL_ALARMS
    FROM DEVICE_REGISTRY dr
    JOIN DEVICE_TELEMETRY dt ON dr.DEVICE_ID = dt.DEVICE_ID
    LEFT JOIN ALARMS a ON dr.DEVICE_ID = a.DEVICE_ID
    WHERE dr.STATUS = 'Active'
    GROUP BY 1, 2, 3, 4
""")

print(f"Feature rows: {telemetry_features.count()}")
telemetry_features.show(5)

In [ ]:
# Future: Train XGBoost model for failure prediction using Snowflake ML
#
# from snowflake.ml.modeling.xgboost import XGBClassifier
#
# model = XGBClassifier(
#     input_cols=["AVG_VIBRATION", "AVG_EFFICIENCY", "AVG_TEMPERATURE",
#                 "OPERATING_HOURS", "DAYS_SINCE_SERVICE", "ALARM_COUNT"],
#     label_cols=["FAILURE_WITHIN_30_DAYS"],
#     output_cols=["PREDICTED_FAILURE"]
# )
#
# model.fit(training_data)
# predictions = model.predict(telemetry_features)

print("ML model training placeholder - using rule-based UDFs for initial deployment")

In [ ]:
# Validate rule-based ML functions
failure_predictions = session.sql("SELECT ENERGY_RECOVERY_DB.ML_MODELS.PREDICT_PX_FAILURE() AS PREDICTIONS")
failure_predictions.show()

efficiency_scores = session.sql("SELECT ENERGY_RECOVERY_DB.ML_MODELS.SCORE_ENERGY_EFFICIENCY() AS SCORES")
efficiency_scores.show()

demand_forecast = session.sql("SELECT ENERGY_RECOVERY_DB.ML_MODELS.FORECAST_DEMAND() AS FORECAST")
demand_forecast.show()

health_scores = session.sql("SELECT ENERGY_RECOVERY_DB.ML_MODELS.CALCULATE_EQUIPMENT_HEALTH() AS HEALTH")
health_scores.show()

In [ ]:
session.close()
print("Session closed.")